In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import sys
import json
import shutil
import importlib
from dataclasses import fields

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# Notebook location expected:
# z.parcels_postprocessing/notebooks/simulate_test.ipynb
NB_DIR = Path.cwd()
PROJECT_DIR = NB_DIR.parent

# Add project root so that src is importable as a package
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Load project modules as modules, not individual functions
import scripts.run as prun
import scripts.fieldset as pfs

# Reload during notebook development
importlib.reload(pfs)
importlib.reload(prun)

<module 'scripts.run' from 'c:\\Users\\Jelle Gortemaker\\Documents\\Thesis\\z.parcels_postprocessing\\scripts\\run.py'>

In [ ]:
# ============================================================
# 2. DIRECTORY / CASE SETTINGS
# ============================================================

case_name = "parcels_sea_win_spinusim_l1"
RUN_TITLE_INFO = "parcels_sea_win_spinusim_l1"


DATA_FILE = (NB_DIR / f"../data/input/{case_name}.nc").resolve()
OUT_DIR = (NB_DIR / f"../results/{case_name}").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Could not find input file at:\n{DATA_FILE}")

print(f"Using input file : {DATA_FILE}")
print(f"Using output dir : {OUT_DIR}")


# ============================================================
# 3. PARCELS SETTINGS
# ============================================================

# MITgcm surface velocity output frequency: every 3 hours
DAY_PER_INDEX = 3.0 / 24.0
TIME_STEP_SECONDS = int(DAY_PER_INDEX * 86400)

# Change only this to release at another saved velocity field
RELEASE_TIME_INDEX = 0

# For the first physical benchmark, use actual surface velocity only
LEVEL_INDICES = (0,)

# Runtime
RUNTIME_DAYS = 7.0
DT_SECONDS = 300
OUTPUTDT_SECONDS = TIME_STEP_SECONDS

# Uniform release grid
NX = 60
NY = 60

# Periodic MITgcm domain
PERIODIC = True

# Avoid zarr overwrite issues
OVERWRITE_OUTPUT = True

OUT_ZARR = OUT_DIR / f"trajectories_{case_name}_k0_release_t{RELEASE_TIME_INDEX:04d}.zarr"

if OVERWRITE_OUTPUT and OUT_ZARR.exists():
    shutil.rmtree(OUT_ZARR)

release_time_days = RELEASE_TIME_INDEX * DAY_PER_INDEX

print(f"release index : {RELEASE_TIME_INDEX}")
print(f"release time  : {release_time_days:.3f} days")
print(f"time step     : {TIME_STEP_SECONDS} s")
print(f"outputdt      : {OUTPUTDT_SECONDS} s")
print(f"release grid  : {NX} x {NY}")


# ============================================================
# 4. CHECK WHETHER UPDATED RUNCONFIG IS AVAILABLE
# ============================================================

required_runconfig_fields = {
    "time_step_seconds",
    "release_time_index",
    "periodic",
    "level_indices",
}

available_runconfig_fields = {f.name for f in fields(prun.RunConfig)}
missing = required_runconfig_fields - available_runconfig_fields

if missing:
    raise RuntimeError(
        "RunConfig is missing fields needed for the updated Parcels workflow:\n"
        f"{sorted(missing)}\n\n"
        "Update scripts/run.py so it supports release_time_index, time_step_seconds, "
        "periodic, and level_indices."
    )


# ============================================================
# 5. RAW DATASET QC
# ============================================================

print("\nOpening raw MITgcm dataset...")
ds_raw = xr.open_dataset(DATA_FILE, chunks="auto")
display(ds_raw)

print("\nRaw dataset summary:")
summary = pfs.summarize_dataset(DATA_FILE)
print(json.dumps(summary, indent=2, default=str))


# ============================================================
# 6. BUILD PARCELS FIELDSET QC
# This should convert:
# UVEL(T, Z, Y, Xp1)  -> U(time, y, x)
# VVEL(T, Z, Yp1, X) -> V(time, y, x)
# ============================================================

print("\nBuilding Parcels fieldset...")

fieldset, meta, ds_parcels = pfs.build_fieldset(
    DATA_FILE,
    surface_only=True,
    mesh="flat",
    level_indices=LEVEL_INDICES,
    time_step_seconds=TIME_STEP_SECONDS,
    periodic=PERIODIC,
)

display(ds_parcels)

print("\nParcels-ready dataset QC:")

try:
    qc = pfs.quick_qc_parcels_input(ds_parcels)
except TypeError:
    # fallback if your older helper still expects explicit variable names
    qc = pfs.quick_qc_parcels_input(ds_parcels, "U", "V")

print(json.dumps(qc, indent=2, default=str))

print("\nParcels time coordinate head:")
print(ds_parcels["time"].values[:8])

print("\nParcels dt head:")
print(np.diff(ds_parcels["time"].values[:8]))


# ============================================================
# 7. RUN PARCELS
# ============================================================

config = prun.RunConfig(
    input_nc=str(DATA_FILE),
    output_path=str(OUT_ZARR),
    runtime_days=RUNTIME_DAYS,
    dt_seconds=DT_SECONDS,
    outputdt_seconds=OUTPUTDT_SECONDS,
    time_step_seconds=TIME_STEP_SECONDS,
    release_time_index=RELEASE_TIME_INDEX,
    surface_only=True,
    mesh="flat",
    periodic=PERIODIC,
    level_indices=LEVEL_INDICES,
    release_mode="grid",
    nx=NX,
    ny=NY,
    release_margin_cells=1.0,
)

print("\nRunning Parcels experiment...")
info = prun.run_parcels_experiment(config)
print(json.dumps(info, indent=2, default=str))


# ============================================================
# 8. LOAD AND QUICK-PLOT TRAJECTORIES
# ============================================================

print("\nLoading trajectories...")
traj = prun.load_trajectories(OUT_ZARR)
display(traj)

xname = "lon" if "lon" in traj.variables else "x"
yname = "lat" if "lat" in traj.variables else "y"

plt.figure(figsize=(7, 6))

n_plot = min(200, traj.sizes["trajectory"])

for i in range(n_plot):
    plt.plot(
        traj[xname].isel(trajectory=i),
        traj[yname].isel(trajectory=i),
        lw=0.6,
        alpha=0.5,
    )

plt.xlabel("x [m]")
plt.ylabel("y [m]")
plt.title(
    f"Particle trajectories\n"
    f"{RUN_TITLE_INFO}; release t = {release_time_days:.2f} days; "
    f"levels={LEVEL_INDICES}"
)
plt.axis("equal")
plt.tight_layout()
plt.show()

FileNotFoundError: Could not find input file at:
C:\Users\Jelle Gortemaker\Documents\Thesis\z.parcels_postprocessing\data\input\parcels_sea_win_spinusim_l1.nc